# Generating QRC inputs from ORCA 5 frequency calculations

pyQRC reads a completed frequency calculation and writes a new input file whose geometry has been displaced along one or more normal modes — Silva and Goodman's *Quick Reaction Coordinate* (QRC) approach. This notebook walks through the ORCA 5 example files that ship in this directory.

Requirements: `pip install pyqrc` (pulls in cclib and numpy).


## Setup

pyQRC writes its new input files next to the file it reads, so we copy the example outputs into a `scratch/` subdirectory and run everything there. The helper below invokes the same `pyqrc` command line you would use in a terminal or HPC batch script.

In [1]:
import shutil
import subprocess
import sys
from pathlib import Path

HERE = Path.cwd()                # this examples directory
SCRATCH = HERE / "scratch"
if SCRATCH.exists():
    shutil.rmtree(SCRATCH)
SCRATCH.mkdir()

for name in ['acetaldehyde.out', 'claisen_ts.out']:
    shutil.copy(HERE / name, SCRATCH / name)


def run_pyqrc(*args):
    """Run the pyqrc command line inside the scratch directory."""
    result = subprocess.run(
        [sys.executable, "-m", "pyqrc", *args],
        cwd=SCRATCH, capture_output=True, text=True,
    )
    print(result.stdout, end="")
    if result.returncode != 0:
        print(result.stderr, end="")
        raise RuntimeError(f"pyqrc exited with code {result.returncode}")


def show(filename, n=40):
    """Print up to n lines of a file in the scratch directory."""
    lines = (SCRATCH / filename).read_text().splitlines()
    print("\n".join(lines[:n]))
    if len(lines) > n:
        print(f"... ({len(lines) - n} more lines)")


## Example 1: remove an unwanted imaginary frequency

This acetaldehyde optimization inadvertently produced a saddle point — it has one small imaginary frequency. By default pyQRC displaces along **all** imaginary modes, which is exactly what we want here: the displaced geometry breaks the symmetry of the saddle point, and re-optimizing it gives the true minimum.

In [2]:
run_pyqrc("acetaldehyde.out", "--nproc", "4", "--mem", "8GB")

o   acetaldehyde.out had 1 imaginary frequencies: processed


That wrote two files: `acetaldehyde_QRC.inp` (the new, displaced input — ready to submit) and `acetaldehyde_QRC.qrc` (a human-readable summary of the frequencies and the displacement).

In [3]:
show("acetaldehyde_QRC.inp")

! M062X D3Zero def2-TZVP Opt Freq
 %pal nprocs 4 end
 %maxcore 8192

# acetaldehyde_QRC

* xyz 0 1
 C   0.23613500   0.41153820  -0.01860840
 O   1.21078120  -0.28630260   0.01507700
 H   0.34334620   1.51367680  -0.08137280
 C  -1.16782600  -0.13487600   0.00507740
 H  -1.91392200   0.65654420   0.10356580
 H  -1.34213100  -0.68217640  -0.92940560
 H  -1.26209480  -0.84900940   0.82949880
*


In [4]:
show("acetaldehyde_QRC.qrc", n=24)

 pyQRC - a quick alternative to IRC calculations
 version: 2.3.0 / author: Robert Paton / email: robert.paton@colostate.edu
 Based on: Goodman, J. M.; Silva, M. A. Tet. Lett. 2003, 44, 8233-8236;
 Tet. Lett. 2005, 46, 2067-2069.

                -----ORIGINAL GEOMETRY------
                       X         Y         Z
   C            0.236135  0.411539  0.000002
   O            1.210781 -0.286303  0.000021
   H            0.343347  1.513679 -0.000026
   C           -1.167826 -0.134876  0.000003
   H           -1.913926  0.656550  0.000080
   H           -1.302134 -0.765529 -0.879503
   H           -1.302086 -0.765667  0.879416

                ----HARMONIC FREQUENCIES----
                    Freq  Red mass   F const
               -200.7700    0.0000    0.0000
                511.7500    0.0000    0.0000
                743.3100    0.0000    0.0000
                945.2100    0.0000    0.0000
               1101.2000    0.0000    0.0000
               1140.6100    0.0000    0.0000
    

## Example 2: map a reaction coordinate (the namesake QRC)

For a transition state — here a Claisen rearrangement — the quick alternative to an IRC is two displaced inputs: one along the imaginary mode (`--amp 0.3`) and one in the reverse direction (`--amp -0.3`). Optimizing both gives the reactant and product the TS connects. The benchmark in the README found an amplitude of **0.3** performs best, hence the values used here; `--name` controls the suffix of the generated files.

In [5]:
run_pyqrc("claisen_ts.out", "--nproc", "4", "--mem", "8GB", "--amp", "0.3", "--name", "QRCF")
run_pyqrc("claisen_ts.out", "--nproc", "4", "--mem", "8GB", "--amp", "-0.3", "--name", "QRCR")

o   claisen_ts.out had 1 imaginary frequencies: processed
o   claisen_ts.out had 1 imaginary frequencies: processed


In [6]:
show("claisen_ts_QRCF.inp", n=14)
print("=" * 60)
show("claisen_ts_QRCR.inp", n=14)

! wB97X-D3 def2-SVP OptTS Freq
 %pal nprocs 4 end
 %maxcore 8192

# claisen_ts_QRCF

* xyz 0 1
 C  -1.30253020   0.85112420  -0.26100730
 C  -1.26730120  -0.49404660   0.26503650
 O  -0.59292170  -1.36580820  -0.24281670
 C   1.40418360  -0.85803340   0.19334720
 C   1.36552830   0.38861970  -0.30489760
 C   0.49705980   1.38110140   0.29484280
 H  -2.02253680   1.54681610   0.18829810
... (8 more lines)
! wB97X-D3 def2-SVP OptTS Freq
 %pal nprocs 4 end
 %maxcore 8192

# claisen_ts_QRCR

* xyz 0 1
 C  -1.52173780   0.73999580  -0.30649870
 C  -1.27089880  -0.43735740   0.26926350
 O  -0.37508630  -1.32302580  -0.21404730
 C   1.11507240  -0.94904260   0.15396680
 C   1.34987370   0.43435230  -0.30356440
 C   0.74243220   1.43125060   0.33650320
 H  -2.18287120   1.47038990   0.16658590
... (8 more lines)


## Using the Python API instead of the CLI

The same machinery is importable. `QRCGenerator` parses the output, computes the displaced geometry, and (unless `write=False`) writes the files in one go. With `write=False` you can inspect the displacement before committing anything to disk.

In [7]:
import numpy as np
from pyqrc import QRCGenerator

qrc = QRCGenerator(
    file=str(SCRATCH / "claisen_ts.out"),
    amplitude=0.3,
    nproc=4,
    mem="8GB",
    route=None,     # None clones the route/keywords from the original job
    verbose=False,  # skip the .qrc summary file
    suffix="API",
    val=None,       # or displace along the mode nearest this frequency (cm-1)
    num=None,       # or along this 1-indexed mode number
    write=False,    # compute only; no files are written
)

freqs = np.asarray(qrc.FREQS)
print("Imaginary frequencies (cm-1):", freqs[freqs < 0.0])
print(f"Mass-weighted displacement from the TS: {qrc.MW_DISTANCE:.4f} bohr amu^1/2")
print("Displaced geometry has clashing atoms:", qrc.OVERLAPPED)
print()

per_atom = np.linalg.norm(qrc.NEW_CARTESIAN - qrc.CARTESIAN, axis=1)
print("Atoms that move the most:")
for i in np.argsort(per_atom)[::-1][:5]:
    print(f"  atom {i + 1:>2} ({qrc.ATOMTYPES[i]:<2}) moved {per_atom[i]:.3f} Angstrom")


Imaginary frequencies (cm-1): [-633.34]
Mass-weighted displacement from the TS: 1.7927 bohr amu^1/2
Displaced geometry has clashing atoms: False

Atoms that move the most:
  atom  4 (C ) moved 0.153 Angstrom
  atom  6 (C ) moved 0.127 Angstrom
  atom  1 (C ) moved 0.125 Angstrom
  atom  3 (O ) moved 0.112 Angstrom
  atom  7 (H ) moved 0.089 Angstrom


## Where the files went

Everything generated above is in the `scratch/` subdirectory (ignored by git) — delete it when you are done. In real use you would submit the new input file (`*.inp`) to your scheduler and optimize.

See the [project README](../../README.md) for the full option list and for the IRC-comparison benchmark behind the recommended amplitude of 0.3.
